# WorldSim DGX Spark QLoRA Smoke Run

Thin notebook wrapper around the shared `training.lib.qlora_smoke` API.


In [ ]:
from training.lib.qlora_smoke import get_environment_summary

environment = get_environment_summary()
environment


In [ ]:
from training.lib.qlora_smoke import get_environment_summary

preflight = get_environment_summary()
torch_info = preflight.get("torch", {})
bnb_info = preflight.get("bitsandbytes", {})

assert torch_info.get("cuda_available"), "CUDA is unavailable in this Jupyter kernel. Run this notebook on a DGX Spark CUDA kernel."
assert bnb_info.get("available"), "bitsandbytes is unavailable. Install requirements-training.txt in the notebook environment."
preflight


In [ ]:
from datetime import UTC, datetime
from pathlib import Path

from training.lib.qlora_smoke import SmokeRunConfig

run_id = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
config = SmokeRunConfig(
    train_file=Path("data/training/worldsim-v31-mix-v1/train_converted.jsonl"),
    dev_file=Path("data/training/worldsim-v31-mix-v1/dev_converted.jsonl"),
    output_dir=Path("outputs/smoke_cuda_notebook/worldsim-v31-mix-v1") / run_id,
    max_steps=5,
    max_train_samples=64,
    max_eval_samples=32,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,
    require_qlora=True,
)
config.to_dict()


In [ ]:
from training.lib.qlora_smoke import run_smoke_or_raise

result = run_smoke_or_raise(config)
result.to_dict()


In [ ]:
from training.lib.qlora_smoke import preview_metrics

metrics = preview_metrics(result.output_dir)
metrics


In [ ]:
from training.lib.qlora_smoke import count_parseable_json_samples, load_sample_generations

samples = load_sample_generations(result.output_dir)
{
    "sample_count": len(samples),
    **count_parseable_json_samples(samples),
}


In [ ]:
samples[:3]


In [ ]:
{
    "status": result.status,
    "used_true_qlora": result.used_true_qlora,
    "output_dir": result.output_dir,
    "summary_path": result.summary_path,
    "metrics_path": result.metrics_path,
    "sample_path": result.sample_path,
    "adapter_dir": result.adapter_dir,
}
